# 🔀 Ensemble Model - Plant Disease Classification

**Strategy:** Combine predictions from multiple diverse models

**Performance:** 68% F1 on PlantDoc (from paper)

**Models in Ensemble:**
1. ResNet50 (CNN baseline)
2. EfficientNet-B3 (Efficient CNN)
3. ViT-Small (Vision Transformer)
4. Swin-Tiny (Hierarchical Transformer)

**Ensemble Method:** Soft voting (average probabilities)

---

## ⚠️ Enable GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
import torch
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install -q timm albumentations scikit-learn opendatasets
print("✅ Installed")

In [ ]:
import os, opendatasets as od
if not os.path.exists('/content/plantvillage'):
    od.download("https://www.kaggle.com/datasets/mohitsingh1804/plantvillage")
if not os.path.exists('/content/PlantDoc-Dataset'):
    !git clone https://github.com/pratikkayal/PlantDoc-Dataset.git
print("✅ Datasets ready")

In [ ]:
import os, random, numpy as np, pandas as pd, cv2, json, time, warnings
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

pv_base = Path('/content/plantvillage/PlantVillage') if Path('/content/plantvillage/PlantVillage').exists() else Path('/content/plantvillage')
PV_TRAIN, PV_VAL, PD_TEST = pv_base/'train', pv_base/'val', Path('/content/PlantDoc-Dataset/test')

BATCH_SIZE, EPOCHS, LR, IMG_SIZE = 32, 15, 1e-4, 224
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(42); np.random.seed(42); torch.manual_seed(42)

In [ ]:
common_labels = sorted([
    "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
    "Blueberry___healthy", "Cherry_(including_sour)___healthy", "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight", "Grape___healthy", "Peach___healthy",
    "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy", "Potato___Early_blight",
    "Potato___Late_blight", "Potato___healthy", "Raspberry___healthy", "Soybean___healthy",
    "Squash___Powdery_mildew", "Strawberry___Leaf_scorch", "Strawberry___healthy",
    "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight",
    "Tomato___Leaf_Mold", "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites_Two-spotted_spider_mite",
    "Tomato___Target_Spot", "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus", "Tomato___healthy"
])

plantdoc_to_plantvillage = {
    "Apple leaf": "Apple___healthy", "Apple rust leaf": "Apple___Cedar_apple_rust",
    "Apple Scab Leaf": "Apple___Apple_scab", "Bell_pepper leaf": "Pepper,_bell___healthy",
    "Bell_pepper leaf spot": "Pepper,_bell___Bacterial_spot", "Blueberry leaf": "Blueberry___healthy",
    "Cherry leaf": "Cherry_(including_sour)___healthy", "Corn Gray leaf spot": "Corn_(maize)___Northern_Leaf_Blight",
    "Corn leaf blight": "Corn_(maize)___Northern_Leaf_Blight", "Corn rust leaf": "Corn_(maize)___Common_rust_",
    "Peach leaf": "Peach___healthy", "Potato leaf": "Potato___healthy",
    "Potato leaf early blight": "Potato___Early_blight", "Potato leaf late blight": "Potato___Late_blight",
    "Raspberry leaf": "Raspberry___healthy", "Soybean leaf": "Soybean___healthy",
    "Squash Powdery mildew leaf": "Squash___Powdery_mildew", "Strawberry leaf": "Strawberry___healthy",
    "Tomato Early blight leaf": "Tomato___Early_blight", "Tomato Septoria leaf spot": "Tomato___Septoria_leaf_spot",
    "Tomato leaf": "Tomato___healthy", "Tomato leaf bacterial spot": "Tomato___Bacterial_spot",
    "Tomato leaf late blight": "Tomato___Late_blight", "Tomato leaf mosaic virus": "Tomato___Tomato_mosaic_virus",
    "Tomato leaf yellow virus": "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato mold leaf": "Tomato___Leaf_Mold",
    "Tomato two spotted spider mites leaf": "Tomato___Spider_mites_Two-spotted_spider_mite",
    "grape leaf": "Grape___healthy", "grape leaf black rot": "Apple___Black_rot"
}

label2idx = {l: i for i, l in enumerate(common_labels)}
num_classes = len(common_labels)
print(f"Classes: {num_classes}")

In [ ]:
class Augmentation:
    def __init__(self, img_size=224, is_train=False):
        if is_train:
            self.t = A.Compose([
                A.Resize(img_size, img_size),
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.3),
                A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
                ToTensorV2()
            ])
        else:
            self.t = A.Compose([
                A.Resize(img_size, img_size),
                A.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
                ToTensorV2()
            ])
    def __call__(self, img):
        return self.t(image=np.array(img) if isinstance(img, Image.Image) else img)['image']

class PVDataset(Dataset):
    def __init__(self, root, l2i, size=224, is_train=False):
        self.t = Augmentation(size, is_train)
        self.samples = [(str(p), l2i[l]) for l in l2i for ext in ['*.jpg','*.JPG','*.png'] 
                       for p in (Path(root)/l).glob(ext) if (Path(root)/l).exists()]
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, l = self.samples[i]
        return self.t(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)), l

class PDDataset(Dataset):
    def __init__(self, root, l2i, pd2pv, size=224):
        self.t = Augmentation(size, is_train=False)
        self.samples = [(str(p), l2i[pv]) for pd, pv in pd2pv.items() if pv in l2i 
                       for ext in ['*.jpg','*.JPG','*.png'] for p in (Path(root)/pd).glob(ext) 
                       if (Path(root)/pd).exists()]
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, l = self.samples[i]
        return self.t(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)), l

In [ ]:
trn = PVDataset(str(PV_TRAIN), label2idx, IMG_SIZE, is_train=True)
val = PVDataset(str(PV_VAL), label2idx, IMG_SIZE, is_train=False)
tst = PDDataset(str(PD_TEST), label2idx, plantdoc_to_plantvillage, IMG_SIZE)

trn_l = DataLoader(trn, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_l = DataLoader(val, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
tst_l = DataLoader(tst, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(trn):,} | Val: {len(val):,} | Test: {len(tst):,}")

In [ ]:
# Create ensemble models
print("Creating ensemble models...\n")

models_dict = {
    'ResNet50': timm.create_model('resnet50', pretrained=True, num_classes=num_classes),
    'EfficientNet-B3': timm.create_model('efficientnet_b3', pretrained=True, num_classes=num_classes),
    'ViT-Small': timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=num_classes),
    'Swin-Tiny': timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=num_classes)
}

for name, model in models_dict.items():
    models_dict[name] = model.to(DEVICE)
    print(f"✅ {name} loaded")

print(f"\n✅ {len(models_dict)} models ready for ensemble")

In [ ]:
def train_epoch(m, loader, crit, opt, dev):
    m.train(); loss_sum, corr, tot = 0, 0, 0
    for img, lbl in tqdm(loader, desc='Train', leave=False):
        img, lbl = img.to(dev), lbl.to(dev)
        opt.zero_grad(); out = m(img); loss = crit(out, lbl)
        loss.backward(); opt.step()
        loss_sum += loss.item(); tot += lbl.size(0)
        corr += out.max(1)[1].eq(lbl).sum().item()
    return loss_sum/len(loader), 100.*corr/tot

def evaluate(m, loader, crit, dev):
    m.eval(); loss_sum, preds, labels = 0, [], []
    with torch.no_grad():
        for img, lbl in tqdm(loader, desc='Eval', leave=False):
            img, lbl = img.to(dev), lbl.to(dev)
            out = m(img); loss_sum += crit(out, lbl).item()
            preds.extend(out.max(1)[1].cpu().numpy())
            labels.extend(lbl.cpu().numpy())
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted', zero_division=0)
    return {'loss': loss_sum/len(loader), 'acc': acc*100, 'f1': f1*100, 'precision': p*100, 'recall': r*100}

def ensemble_evaluate(models, loader, dev):
    """Evaluate ensemble using soft voting"""
    for m in models.values():
        m.eval()
    
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for img, lbl in tqdm(loader, desc='Ensemble Eval'):
            img, lbl = img.to(dev), lbl.to(dev)
            
            # Get predictions from all models
            ensemble_probs = []
            for model in models.values():
                out = model(img)
                probs = F.softmax(out, dim=1)
                ensemble_probs.append(probs)
            
            # Average probabilities (soft voting)
            avg_probs = torch.stack(ensemble_probs).mean(dim=0)
            preds = avg_probs.max(1)[1]
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(lbl.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    p, r, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    return {'acc': acc*100, 'f1': f1*100, 'precision': p*100, 'recall': r*100}

In [ ]:
# Train each model individually
print("\n" + "="*60)
print("Training Individual Models")
print("="*60 + "\n")

crit = nn.CrossEntropyLoss()
total_time = 0

for model_name, model in models_dict.items():
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}\n")
    
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3)
    
    best_f1, patience_counter = 0, 0
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch+1}/{EPOCHS}")
        tl, ta = train_epoch(model, trn_l, crit, opt, DEVICE)
        v = evaluate(model, val_l, crit, DEVICE)
        sch.step(v['f1'])
        
        print(f"Loss: {tl:.4f} | Val F1: {v['f1']:.2f}%")
        
        if v['f1'] > best_f1:
            best_f1 = v['f1']; patience_counter = 0
            torch.save(model.state_dict(), f'/content/{model_name}.pth')
            print(f"✓ Best: {best_f1:.2f}%")
        else:
            patience_counter += 1
            if patience_counter >= 5:
                print(f"Early stop"); break
    
    model.load_state_dict(torch.load(f'/content/{model_name}.pth'))
    total_time += time.time() - t0
    print(f"✅ {model_name} training complete\n")

In [ ]:
# Evaluate ensemble
print("\n" + "="*60)
print("Evaluating Ensemble")
print("="*60 + "\n")

test = ensemble_evaluate(models_dict, tst_l, DEVICE)

print(f"\n{'='*60}")
print("FINAL ENSEMBLE RESULTS")
print(f"{'='*60}")
print(f"Test Accuracy: {test['acc']:.2f}%")
print(f"Test F1: {test['f1']:.2f}%")
print(f"Test Precision: {test['precision']:.2f}%")
print(f"Test Recall: {test['recall']:.2f}%")
print(f"Total Training Time: {total_time/60:.1f} min")
print(f"{'='*60}")

In [ ]:
results = {
    'model': 'Ensemble (ResNet50 + EfficientNet-B3 + ViT-Small + Swin-Tiny)',
    'num_models': len(models_dict),
    'ensemble_method': 'Soft Voting (Average Probabilities)',
    'test_acc': test['acc'],
    'test_f1': test['f1'],
    'test_precision': test['precision'],
    'test_recall': test['recall'],
    'total_time_minutes': total_time/60
}

with open('/content/ensemble_results.json', 'w') as f:
    json.dump(results, f, indent=2)

fig, ax = plt.subplots(figsize=(8, 5))
metrics = ['Accuracy', 'F1', 'Precision', 'Recall']
values = [test['acc'], test['f1'], test['precision'], test['recall']]
ax.bar(metrics, values, color='purple', alpha=0.8)
ax.set_ylabel('Score (%)', fontweight='bold')
ax.set_title('Ensemble Test Performance (PlantDoc)\n4 Models - Soft Voting', fontweight='bold')
ax.set_ylim([0, 100])
for i, v in enumerate(values):
    ax.text(i, v+2, f'{v:.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('/content/ensemble_results.png', dpi=300)
plt.show()

print("\n✅ Saved: ensemble_results.json, ensemble_results.png")

In [ ]:
from google.colab import files
files.download('/content/ensemble_results.json')
files.download('/content/ensemble_results.png')
print("✅ Downloaded!")